## 12_task_c — Job task: summarize practice_customers_enriched -> practice_customers_summary

Used by: practice-job-2-multitask (task c), practice-job-4b-target
Config: CATALOG/SCHEMA/PARTITION/CONFIG_PATH widgets -> setup_config() (C360-style)
PARTITION: all | latest | prev | YYYY-MM-DD


In [ ]:

%run ../lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")
dbutils.widgets.text("PARTITION", "all", "all | latest | prev | YYYY-MM-DD")
dbutils.widgets.text("CONFIG_PATH", "", "Optional env config file (empty = built-in dev/stg/prd)")

CFG = setup_config(
    config_path=dbutils.widgets.get("CONFIG_PATH").strip(),
    run_date=dbutils.widgets.get("PARTITION").strip(),
    schema=dbutils.widgets.get("SCHEMA").strip(),
)


In [ ]:

from pyspark.sql import functions as F

src = CFG.table("practice_customers_enriched")

summary = (
    CFG.partition_filter(spark.read.table(src), table=src)
    .groupBy("segment", "ingest_date")
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("monthly_revenue"), 2).alias("avg_monthly_revenue"),
        F.round(F.sum("annual_revenue"), 2).alias("total_annual_revenue"),
        F.round(F.avg("churn_risk"), 4).alias("avg_churn_risk"),
    )
    .orderBy("segment", "ingest_date")
)

CFG.write_partitioned(summary, "practice_customers_summary")
display(summary)
print(f"TASK C DONE — wrote summary to {CFG.table('practice_customers_summary')} (PARTITION={CFG.run_date})")
